In [1]:
!pip install torch

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/122.1 MB 548.6 kB/s eta 0:03:42
   ---------------------------------------- 0.5/122.1 MB 548.6 kB/s eta 0:03:42
   ---------------------------------------- 0.8/122.1 MB 583.7 kB/s eta 0:03:28
   ---------------------------------------- 0.8/122.1 MB 583.7 kB/s eta 0:03:28
   ---------------------------------------- 1.0/122.1 MB 607.0 kB/s eta 0:03:20
   ---------------------------------------- 1.0/122.1 MB 607.0 kB/s eta 0:03:20
   ---------------------------------------- 1.3/122.1 MB 619.0 kB/s eta 0:03:16
   ----

In [2]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from src.mlflow_setup import init_tracking
from src.features import build_features
from src.data_split import time_based_split, wmae
import mlflow

init_tracking()

train = pd.read_csv('../data/train.csv')
features = pd.read_csv('../data/features.csv')
stores = pd.read_csv('../data/stores.csv')

df = build_features(train, features, stores)
print(df.shape)

C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(421570, 22)


DLinear (Decomposition Linear) decomposes a time series into trend and seasonal/remainder components using a moving average, then fits a single linear layer to each component separately, summing their outputs for the final forecast. Despite its simplicity, it was shown to outperform many Transformer-based models on standard forecasting benchmarks, challenging the assumption that complex attention mechanisms are necessary for time series. It works on fixed-length input/output windows per series, unlike tree-based models which use engineered point-in-time features.

In [3]:
pivot = df.pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales', aggfunc='sum')
pivot = pivot.sort_index(axis=1)
print(pivot.shape)

(3331, 143)


In [4]:
pivot_filled = pivot.fillna(0)

lookback = 52
horizon = 10

X_windows = []
y_windows = []

data = pivot_filled.values

for i in range(data.shape[0]):
    series = data[i]
    for start in range(0, len(series) - lookback - horizon + 1):
        X_windows.append(series[start:start+lookback])
        y_windows.append(series[start+lookback:start+lookback+horizon])

X_windows = np.array(X_windows)
y_windows = np.array(y_windows)
print(X_windows.shape, y_windows.shape)

(273142, 52) (273142, 10)


In [5]:
split_idx = int(len(X_windows) * 0.9)
X_train_dl, X_val_dl = X_windows[:split_idx], X_windows[split_idx:]
y_train_dl, y_val_dl = y_windows[:split_idx], y_windows[split_idx:]

X_train_t = torch.tensor(X_train_dl, dtype=torch.float32)
y_train_t = torch.tensor(y_train_dl, dtype=torch.float32)
X_val_t = torch.tensor(X_val_dl, dtype=torch.float32)
y_val_t = torch.tensor(y_val_dl, dtype=torch.float32)

print(X_train_t.shape, X_val_t.shape)

torch.Size([245827, 52]) torch.Size([27315, 52])


In [6]:
class DLinear(nn.Module):
    def __init__(self, lookback, horizon, kernel_size=25):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg_pool = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=kernel_size // 2)
        self.linear_trend = nn.Linear(lookback, horizon)
        self.linear_seasonal = nn.Linear(lookback, horizon)

    def forward(self, x):
        x_ = x.unsqueeze(1)
        trend = self.avg_pool(x_).squeeze(1)
        trend = trend[:, :x.shape[1]]
        seasonal = x - trend
        out = self.linear_trend(trend) + self.linear_seasonal(seasonal)
        return out

model_dl = DLinear(lookback=52, horizon=10)
print(model_dl)

DLinear(
  (avg_pool): AvgPool1d(kernel_size=(25,), stride=(1,), padding=(12,))
  (linear_trend): Linear(in_features=52, out_features=10, bias=True)
  (linear_seasonal): Linear(in_features=52, out_features=10, bias=True)
)


In [9]:
import torch.optim as optim

mlflow.set_experiment("DLinear_Training")

def train_dlinear(model, X_train, y_train, X_val, y_val, epochs=10, batch_size=512, lr=1e-3, run_name="DLinear_baseline"):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({'lookback': 52, 'horizon': 10, 'epochs': epochs, 'batch_size': batch_size, 'lr': lr})

        n = X_train.shape[0]
        for epoch in range(epochs):
            model.train()
            perm = torch.randperm(n)
            total_loss = 0
            for i in range(0, n, batch_size):
                idx = perm[i:i+batch_size]
                xb, yb = X_train[idx], y_train[idx]

                optimizer.zero_grad()
                preds = model(xb)
                loss = loss_fn(preds, yb)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(idx)

            train_loss = total_loss / n

            model.eval()
            with torch.no_grad():
                val_preds = model(X_val)
                val_loss = loss_fn(val_preds, y_val).item()

            mlflow.log_metric("train_mae", train_loss, step=epoch)
            mlflow.log_metric("val_mae", val_loss, step=epoch)
            print(f"Epoch {epoch+1}/{epochs} — train_mae: {train_loss:.2f}, val_mae: {val_loss:.2f}")

        torch.save(model.state_dict(), "dlinear_model.pt")
        mlflow.log_artifact("dlinear_model.pt")

        return val_loss

model_dl = DLinear(lookback=52, horizon=10)
final_val_mae = train_dlinear(model_dl, X_train_t, y_train_t, X_val_t, y_val_t, epochs=10)

Epoch 1/10 — train_mae: 2352.74, val_mae: 1077.24
Epoch 2/10 — train_mae: 1622.91, val_mae: 1001.77
Epoch 3/10 — train_mae: 1559.79, val_mae: 997.30
Epoch 4/10 — train_mae: 1540.48, val_mae: 987.14
Epoch 5/10 — train_mae: 1538.35, val_mae: 982.08
Epoch 6/10 — train_mae: 1535.05, val_mae: 981.64
Epoch 7/10 — train_mae: 1533.62, val_mae: 991.45
Epoch 8/10 — train_mae: 1534.20, val_mae: 974.00
Epoch 9/10 — train_mae: 1534.62, val_mae: 973.55
Epoch 10/10 — train_mae: 1535.55, val_mae: 993.22
🏃 View run DLinear_baseline at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/4/runs/982603248f0b46228b3428f6255c8230
🧪 View experiment at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/4
